# 11.f — Ad hoc test: where, cell by cell, is the effect of the AMOC weakening on the bad seasons beyond the variability?

**11.f = the code of 11.e (11.a, sections 1–5, unchanged, and section 6, the zones) + section 7, cell by cell**: the
season-by-season events exist only in memory, so the code is run again here, in a separate notebook and folder
(`<heat folder>_cells/`, e.g. `heat_physio_cells/`); 11.a, 11.e and their outputs are not touched. Default
`RISK_HEAT=physio`. Section 7 does for the bad seasons what 12.b does for the suitability. The description below is
that of 11.a.

**Ad hoc test, outside the suitability chain**: it reads the masks of 0.1, the calendars of the ERA5 BIO files of
1.1.d and the daily climate of the configuration, and writes only in `<out_root>/test_risk_indicators/<route>/`.

**Question.** The suitability (SVM) works on 30-year means: a cell can stay suitable on average and see many more bad
seasons. Where the crops are grown today, how often do the bad seasons come, and does the AMOC collapse make them more
or less frequent? Approach of Ceglar et al. (2021, *Environ. Res. Lett.* 16, 104049, sections 2.4 and 3.1, figures 1–2),
on the crops and calendars of this chain.

**Domain.** The cells where the crop is grown today (MIRCA2000 presence of the route, as the labels of 1.1.d), weighted
by the harvested area; the calendar of the ERA5 BIO file (sowing, anthesis = start of phase 2 of variant A, maturity),
fixed in every scenario.

**Indicators, per crop, cell and season.**
- **Heat** (11.a, 11.b): number of days with Tmax above the critical temperature of the crop between anthesis and
  maturity (maize 35 °C, wheat 34 °C, rice 36 °C, soybean 39 °C, Gourdji et al. 2013, *Environ. Res. Lett.* 8, 024041;
  potato 30 °C, provisional). Ceglar et al. count the days above 28 °C for durum wheat.
- **Drought**: SPEI (Vicente-Serrano et al. 2010, *J. Climate* 23, 1696–1718) of the season (sowing → maturity) and of
  phase 2: the water balance Σ (pr − ET0) of the window (ET0 Hargreaves, as 1.1.d), a log-logistic distribution
  fitted per cell on the reference seasons by L-moments (section 2), then the standard normal quantile.

**Events, relative to the reference of each cell** (EC-Earth3 historical 1981–2014, raw): a **hot season** has a heat
indicator above its 75th percentile of the reference seasons (Ceglar et al.); a **drought** has SPEI ≤ −1.5 (severe) or
≤ −2 (extreme). By construction the reference has about 25 % hot seasons (fewer where its 75th percentile is 0) and
6.7 % / 2.3 % droughts. The events are relative to the climate of the model, so a constant bias of EC-Earth3 cancels:
the runs are used **raw**, with their own sequences (persistence, year-to-year variability).

**11.b: threshold mapped to EC-Earth3 (`RISK_HEAT=mapped`).** EC-Earth3 is too hot at the crop thresholds on the
cropland (first run: hot days per season maize 3.5 vs 1.4 in ERA5, rice 1.9 vs 0.7, soybean 0.6 vs 0.05), so an
absolute threshold counts part of its bias. Instead of correcting the data, the threshold is corrected, cell by cell:
the quantile q of the threshold in the daily values of the window in ERA5 (1981–2014, all seasons) gives the threshold
of EC-Earth3 = its own quantile q of the same days of EC-Earth3 historical (quantile mapping of the threshold; the runs
stay raw, with their own sequences). Beyond the 99th (or below the 1st) percentile of ERA5, where the threshold is
rarely or never reached, it is shifted by the difference of that percentile between EC-Earth3 and ERA5. Outputs in
`heat_mapped/`; section 5 compares ERA5 at the threshold with EC-Earth3 historical at its mapped threshold.

**11.c: indicators of the crop physiology (`RISK_HEAT=physio`, thresholds mapped as 11.b).** Two mechanisms are told
apart: the failure of the reproduction (sterility of pollen or flowers after a few hours of heat around anthesis: the
daily Tmax) and the loss of yield during grain or tuber filling (cumulative heat accelerating senescence, or warm
nights through respiration: the Tmin). Where the literature gives a range, the **most tolerant** (highest) threshold
is used, so as not to overstate the risk; `RISK_DT` (±2 °C) shifts every absolute threshold for the sensitivity.

| crop | reproduction (30 days around anthesis) | yield (filling) | literature |
|---|---|---|---|
| maize | days Tmax > 36 °C | degree-days above 30 °C over the season (single-sine hourly curve between Tmin and Tmax) | pollen viability falls above 35–36 °C; yields fall above 29 °C (Schlenker & Roberts 2009, *PNAS* 106, 15594) and with degree-days above 30 °C (Lobell et al. 2011, *Nat. Clim. Change* 1, 42) |
| soybean | — (39 °C almost never reached) | degree-days above 30 °C over the season | yields fall above 30 °C (Schlenker & Roberts 2009); pod set falls above 35–38 °C |
| rice | days Tmax > 38 °C | warm nights: mean Tmin of grain filling (relative, no threshold) | sterility after ≥ 1 h at ≥ 33.7 °C at anthesis (Jagadish et al. 2007, *J. Exp. Bot.* 58, 1627), 35–38 °C depending on the cultivar; yield −10 % per +1 °C of Tmin, Tmax not significant (Peng et al. 2004, *PNAS* 101, 9971) |
| wheat | days Tmax > 31 °C | days Tmax > 34 °C in grain filling | pollination limited above 31 °C (Porter & Gawith 1999, *Eur. J. Agron.* 10, 23); senescence accelerated above 34 °C (Lobell et al. 2012, *Nat. Clim. Change* 2, 186) |
| potato | — | warm nights: days Tmin > 20 °C from tuber initiation (sowing + 50 days) to maturity | tuberization reduced above 18–20 °C at night, stopped above 25 °C; night worse than day (Levy & Veilleux 2007) |

Outputs in `heat_physio/` (`heat_physio_dT+2/`, `heat_physio_dT-2/` for the sensitivity). Limits: air, not canopy,
temperature; thresholds depending on the cultivar; anthesis approximated (maturity − 60 days), hence the 30-day window.

**11.d: the risk on the suitable land (`RISK_DOMAIN=suitable`).** Are the lands that the suitability gains safe, or do
they see many bad seasons? Next to today's cropland (weighted by the harvested area), fixed domains from 6.b (high
input; cells novel in either climate of a comparison left out), weighted by the land area: the land suitable on ERA5,
the land gained and lost from ERA5 in each run, the land gained and lost with the hosing. Outside the cropland the
calendar is the gap-filled one of 1.1.d (5 nearest cropped cells): the dates around anthesis are less reliable there,
so the drought (SPEI) is the firmer indicator. Outputs in `<heat folder>_suitable/`. Memory: the daily arrays cover
every cell of the domains (most of the land for maize), about 3 GB per variable for a 50-year run.

**Outputs.** Per crop and scenario: frequency of the events per cell (maps), area-weighted over today's cropland
(globally and in the 6 AR6 land regions with the largest area of the crop; Iturbide et al. 2020), probability of
exceedance of the share of the cropland affected in a season (figure 2 of Ceglar et al.), hosing − no hosing; the
significance of the hosing effect against the year-to-year variability of the runs (section 6, `risk_significance.csv`).

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters
CHOSEN_VARIANT = "A_60d"          # calendar of the ERA5 BIO files (anthesis = start of phase 2)
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")    # BIO files of 1.1.d (calendar)
REF, REF_PERIOD = "gcm_historical", ("1981", "2014")  # reference of the events: EC-Earth3 historical, raw
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]   # whole files, raw
RUN_PERIOD = {}                   # run -> (first, last year) to read only part of a file; whole file by default
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
HEAT_TCRIT = {"Maize": 35.0, "Wheat": 34.0, "Rice": 36.0, "Soybean": 39.0, "Potato": 30.0}   # degC, 11.a / 11.b
# 11.c: (name, variable, window, kind, threshold degC): kind "days" = days above the threshold, "kdd" = degree-days
# above it (single sine), "mean" = mean of the window (relative event, no threshold); most tolerant threshold of the
# range of the literature (markdown above)
PHYSIO = {
    "Maize": [("sterility (Tmax > 36 degC, anthesis)", "tasmax", "flower", "days", 36.0),
              ("yield heat (degree-days > 30 degC, season)", "tasmax", "season", "kdd", 30.0)],
    "Soybean": [("yield heat (degree-days > 30 degC, season)", "tasmax", "season", "kdd", 30.0)],
    "Rice": [("sterility (Tmax > 38 degC, anthesis)", "tasmax", "flower", "days", 38.0),
             ("warm nights (mean Tmin, grain filling)", "tasmin", "p2", "mean", None)],
    "Wheat": [("sterility (Tmax > 31 degC, anthesis)", "tasmax", "flower", "days", 31.0),
              ("senescence (Tmax > 34 degC, grain filling)", "tasmax", "p2", "days", 34.0)],
    "Potato": [("warm nights (Tmin > 20 degC, tuber bulking)", "tasmin", "tuber", "days", 20.0)],
}
FLOWER_WINDOW = (-15, 15)         # days around anthesis (start of phase 2): 30 days, Gourdji et al. 2013
TUBER_START = 50                  # potato: tuber initiation, days after sowing (variant C_crop of 1.1.d)
HEAT_PCT = 75                     # hot season: heat indicator above this percentile of the reference seasons
SPEI_THR = (-1.5, -2.0)           # drought: SPEI <= threshold (severe, extreme)
N_REGIONS = 6                     # AR6 land regions with the largest area of the crop
ERA5_CHECK = True                 # section 5: heat indicators of ERA5 vs EC-Earth3 historical
RISK_HEAT = os.environ.get("RISK_HEAT", "physio")   # "absolute": Tcrit everywhere (11.a) | "mapped": Tcrit mapped
                                  # cell by cell to EC-Earth3 (11.b) | "physio": PHYSIO, mapped (11.c); environment
RISK_DT = float(os.environ.get("RISK_DT", "0"))       # degC added to every absolute threshold (sensitivity: -2, +2)
TAIL_Q = 0.01                     # mapping: beyond the 1st / 99th percentile of ERA5, shift by the difference of that percentile
BIN = (-60.0, 70.0, 0.1)          # degC: bins of the histograms of the mapping
RISK_DOMAIN = os.environ.get("RISK_DOMAIN", "cropland")   # "cropland": today's cropland (11.a-c) | "suitable": also
                                  # the land suitable on ERA5 and gained / lost (6.b) (11.d); set with the environment
SUIT_LEVEL = "high"               # 11.d: GAEZ input level of the suitable land of 6.b
assert RISK_HEAT in ("absolute", "mapped", "physio"), RISK_HEAT
assert RISK_DOMAIN in ("cropland", "suitable"), RISK_DOMAIN
MAPPED = RISK_HEAT in ("mapped", "physio")

ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
SUB = {"absolute": [], "mapped": ["heat_mapped"], "physio": ["heat_physio"]}[RISK_HEAT]
if RISK_DT:
    SUB = [(SUB[0] if SUB else "heat_absolute") + f"_dT{RISK_DT:+g}"]
if RISK_DOMAIN == "suitable":
    SUB = [(SUB[0] if SUB else "heat_absolute") + "_suitable"]
SUB = [(SUB[0] if SUB else "heat_absolute") + "_cells"]            # 11.f: its own folder, 11.a and 11.e untouched
AMOC_DIR = cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, f"{CHOSEN_VARIANT}_no_rad")   # final suitability of 6.b
OUT = cu.out_dir(CFG, "test_risk_indicators", ROUTE, *SUB)          # the only folder written
SCEN = ["historical", *RUNS]
SPEC = {"historical": (REF, REF_PERIOD), **{r: (r, RUN_PERIOD.get(r)) for r in RUNS}}

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
base_of = lambda crop: crop.split("_")[0].split("-")[0]
# heat indicators of each crop: (name, variable, window, kind, threshold), the threshold shifted by RISK_DT
HEAT = {c: [(n, v, wnd, k, None if t is None else t + RISK_DT) for n, v, wnd, k, t in
            (PHYSIO[base_of(c)] if RISK_HEAT == "physio" else [("heat", "tasmax", "p2", "days", HEAT_TCRIT[base_of(c)])])]
        for c in CROPS}

# domains of each crop: today's cropland (presence of the route, as the labels of 1.1.d) and, for 11.d, the suitable
# land of 6.b; harvested area, land area, calendar (gap-filled off the crop in 1.1.d)
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
CROP = {}
for crop in CROPS:
    cm = cu.crop_mask(CFG, crop)
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        cal = {v: ds[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
    known = LAND & np.isfinite(cal["sowing_doy"]) & np.isfinite(cal["maturity_doy"])
    pres = known & (cm["presence"].values == 1)
    if CELL_AREA_THR:
        pres &= cu.cell_area_frac(CFG, cm["area_ha"].values) > CELL_AREA_THR
    dom = {"cropland today": pres}
    if RISK_DOMAIN == "suitable":
        with xr.open_dataset(AMOC_DIR / f"climate_soil_scenarios_{crop}.nc") as ds:
            S = {str(x): ds[f"suitable_{SUIT_LEVEL}"].sel(scenario=x).values == 1 for x in ds.scenario.values}
            N = {str(x): ds["novel"].sel(scenario=x).values == 1 for x in ds.scenario.values}
        both = lambda a, b: known & ~N[a] & ~N[b]                    # assessable in both climates
        dom["suitable on ERA5"] = known & S["ERA5"]
        for r in [r for r in RUNS if r in S]:
            dom[f"gained, ERA5 -> {r}"] = both("ERA5", r) & S[r] & ~S["ERA5"]
            dom[f"lost, ERA5 -> {r}"] = both("ERA5", r) & S["ERA5"] & ~S[r]
        for h, o in [(h, o) for h, o in PAIRS if h in S and o in S]:
            dom[f"gained with hosing, {h}"] = both(h, o) & S[h] & ~S[o]
            dom[f"lost with hosing, {h}"] = both(h, o) & S[o] & ~S[h]
    CROP[crop] = {"mask": np.logical_or.reduce(list(dom.values())), "dom": dom,
                  "area": np.nan_to_num(cm["area_ha"].values), "cal": cal}
UNION = np.logical_or.reduce([CROP[c]["mask"] for c in CROPS])
UI, UJ = np.nonzero(UNION)
COL = -np.ones(LAND.shape, int); COL[UI, UJ] = np.arange(UI.size)   # column of each cell in the daily arrays
for crop in CROPS:
    i, j = np.nonzero(CROP[crop]["mask"])
    CROP[crop].update({"ij": (i, j), "col": COL[i, j], "w": CROP[crop]["area"][i, j], "wl": LAND_HA[i, j],
                       "dom": {k: v[i, j] for k, v in CROP[crop]["dom"].items()},
                       **{k: CROP[crop]["cal"][k][i, j].astype(float) for k in CROP[crop]["cal"]}})

# AR6 land regions (Iturbide et al. 2020) of the cells
try:
    import regionmask
    AR6 = regionmask.defined_regions.ar6.land
    REG = AR6.mask(GRID["lon"], GRID["lat"]).values                  # (lat, lon), region number or NaN
    REG_NAME = dict(zip(AR6.numbers, AR6.abbrevs))
except Exception as e:                       # no regions: global results only
    print("regionmask not available:", e)
    REG, REG_NAME = np.full(LAND.shape, np.nan), {}
print(f"{len(CROPS)} crops, {UI.size} cells ({RISK_DOMAIN}) | reference {REF} {REF_PERIOD} | runs {RUNS} | heat {RISK_HEAT}, "
      f"thresholds {RISK_DT:+g} degC | written only in {OUT}")
for c in CROPS:
    print(f"  {c:16s} " + " | ".join(h[0] if h[4] is None or not RISK_DT else f"{h[0]} -> {h[4]:g} degC" for h in HEAT[c]))

## 1. Seasonal indicators of every scenario (daily climate read once per scenario, cropland cells only)

In [ ]:
def compute_Ra(lat_rad, doy):
    Gsc = 0.0820
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat_rad) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * Gsc * dr * (ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws))


def eto_hargreaves(tx, tn, lat_rad, doy):
    # as compute_ETo_day of 1.1.d: Ra in mm/day equivalent (divided by the latent heat of vaporisation)
    tm = (tx + tn) / 2.0
    return 0.0023 * compute_Ra(lat_rad, doy) / (2.501 - 0.002361 * tm) * (tm + 17.8) * np.sqrt(np.maximum(tx - tn, 0.0))


def degree_days_above(tx, tn, b):
    # degree-days above b of one day, single-sine curve between Tmin and Tmax (Baskerville & Emin 1969)
    m, a = (tx + tn) / 2, np.maximum((tx - tn) / 2, 1e-6)
    th = np.arcsin(np.clip((b - m) / a, -1, 1))
    return np.maximum(((m - b) * (np.pi / 2 - th) + a * np.cos(th)) / np.pi, 0.0)


def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def windows(dates, c):
    # the seasons of the crop (harvest year hy, with hy - 1 in the data: the seasons crossing New Year start in hy - 1):
    # day indices and masks (days, cells) of the season, phase 2, the 30 days around anthesis, tuber bulking
    yr, doy = dates.year.values, dates.dayofyear.values
    years = sorted(set(yr))
    sow, ant, mat = c["sowing_doy"], c["phase2_start_doy"], c["maturity_doy"]
    wrap = mat > 365
    for hy in years:
        if hy - 1 not in years:
            continue
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = sow[None] + shift, ant[None] + shift, mat[None] + shift
        yield hy, idx, {"season": (T >= s) & (T <= m), "p2": (T >= a) & (T <= m),
                        "flower": (T >= a + FLOWER_WINDOW[0]) & (T < a + FLOWER_WINDOW[1]),
                        "tuber": (T >= s + TUBER_START) & (T <= m)}


def seasons(arr, ET, dates, c, heat, thr):
    # per season and cell of the crop: the heat indicators (thr: one threshold per cell for each) and the water balance
    # of the season and of phase 2 (ET None: no water balance)
    col = c["col"]
    hys, out = [], {k: [] for k in [h[0] for h in heat] + (["D_SEASON", "D_P2"] if ET is not None else [])}
    for hy, idx, W in windows(dates, c):
        X = {v: arr[v][idx][:, col] for v in arr if v != "pr"}
        for name, var, wnd, kind, _ in heat:
            m = W[wnd]
            if kind == "days":
                out[name].append(((X[var] > thr[name]) & m).sum(0))
            elif kind == "kdd":
                out[name].append((degree_days_above(X["tasmax"], X["tasmin"], thr[name]) * m).sum(0))
            else:
                out[name].append(np.where(m, X[var], 0.0).sum(0) / np.maximum(m.sum(0), 1))
        if ET is not None:
            wb = arr["pr"][idx][:, col] - ET[idx][:, col]
            out["D_SEASON"].append(np.where(W["season"], wb, 0.0).sum(0))
            out["D_P2"].append(np.where(W["p2"], wb, 0.0).sum(0))
        hys.append(hy)
    return np.array(hys), {k: np.array(v, float) for k, v in out.items()}


NB = int(round((BIN[1] - BIN[0]) / BIN[2]))


def counts(X, dates, c, wnd):
    # histogram (cells, bins) of the daily values of X in the window, over all the seasons
    n, col = c["col"].size, c["col"]
    cnt = np.zeros(n * NB, np.int64)
    for _, idx, W in windows(dates, c):
        b = np.clip(((X[idx][:, col] - BIN[0]) / BIN[2]).astype(int), 0, NB - 1)
        cnt += np.bincount((np.arange(n)[None] * NB + b)[W[wnd]], minlength=n * NB)
    return cnt.reshape(n, NB)


def cdf(cnt):
    return np.cumsum(cnt, 1) / np.maximum(cnt.sum(1, keepdims=True), 1)


def value_at(cnt, q):
    # value at the quantile q (per cell): upper edge of the first bin where the empirical CDF reaches q
    return BIN[0] + (np.argmax(cdf(cnt) >= np.asarray(q)[:, None] - 1e-12, axis=1) + 1) * BIN[2]


def mapped_threshold(cnt_e, cnt_m, t0):
    # threshold of EC-Earth3 = its quantile q of the window, q = quantile of t0 in ERA5; beyond the 1st / 99th
    # percentile of ERA5: t0 shifted by the difference of that percentile (EC-Earth3 - ERA5)
    n = cnt_e.shape[0]
    q = cdf(cnt_e)[np.arange(n), int(np.clip((t0 - BIN[0]) / BIN[2], 0, NB - 1))]
    thr = value_at(cnt_m, np.clip(q, TAIL_Q, 1 - TAIL_Q))
    for tail, sel in ((1 - TAIL_Q, q > 1 - TAIL_Q), (TAIL_Q, q < TAIL_Q)):
        qq = np.full(n, tail)
        thr = np.where(sel, t0 + value_at(cnt_m, qq) - value_at(cnt_e, qq), thr)
    return thr, q


def read_daily(name, period, vars_=("tasmax", "tasmin", "pr")):
    t0 = time.time()
    arr = {}
    for v in vars_:
        da = cu.read_climate(CFG, name, v, GRID, period)
        arr[v] = cu.land_series(da, (UI, UJ))
    dates = pd.DatetimeIndex(da["time"].values).normalize()
    print(f"  {name} {period or 'whole file'}: {dates[0].date()} -> {dates[-1].date()} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    return arr, dates


LATR = np.radians(GRID["lat"][UI])
TH0 = {c: {h[0]: np.full(CROP[c]["col"].size, np.nan if h[4] is None else h[4]) for h in HEAT[c]} for c in CROPS}
TH = {c: dict(TH0[c]) for c in CROPS}        # thresholds per cell used for EC-Earth3 (mapped below)
if MAPPED or ERA5_CHECK:                     # ERA5: histograms of the mapping and indicators of the check (section 5)
    arr, dates = read_daily("era5", ERA5_PERIOD, ("tasmax", "tasmin"))
    CNT_E = {(c, h[0]): counts(arr[h[1]], dates, CROP[c], h[2]) for c in CROPS for h in HEAT[c] if h[4] is not None} if MAPPED else {}
    E5 = {c: seasons(arr, None, dates, CROP[c], HEAT[c], TH0[c])[1] for c in CROPS} if ERA5_CHECK else {}
    del arr
SEAS = {}                                    # (scenario, crop) -> (harvest years, {indicator: (seasons, cells)})
for s in SCEN:
    name, period = SPEC[s]
    arr, dates = read_daily(name, period)
    if s == "historical" and MAPPED:
        rows = []
        for crop in CROPS:
            w = CROP[crop]["w"]
            for h in HEAT[crop]:
                if h[4] is None:
                    continue
                thr, q = mapped_threshold(CNT_E[(crop, h[0])], counts(arr[h[1]], dates, CROP[crop], h[2]), h[4])
                TH[crop][h[0]] = thr
                rows.append({"crop": crop, "indicator": h[0], "threshold (degC)": h[4],
                             "mapped threshold, area mean (degC)": np.average(thr, weights=w),
                             "shift, area p10 (degC)": float(np.percentile(thr - h[4], 10)),
                             "shift, area p90 (degC)": float(np.percentile(thr - h[4], 90)),
                             "quantile of the threshold in ERA5, area mean": np.average(q, weights=w),
                             f"area beyond the ERA5 p{100 * (1 - TAIL_Q):g} (tail shift, %)": 100 * w[q > 1 - TAIL_Q].sum() / w.sum()})
        THR = pd.DataFrame(rows).set_index(["crop", "indicator"])
        THR.to_csv(OUT / "heat_threshold_mapped.csv")
        display(THR.round(2))
    doys = dates.dayofyear.values
    ET = np.empty_like(arr["tasmax"])
    for t in range(dates.size):
        ET[t] = eto_hargreaves(arr["tasmax"][t], arr["tasmin"][t], LATR, doys[t])
    for crop in CROPS:
        SEAS[(s, crop)] = seasons(arr, ET, dates, CROP[crop], HEAT[crop], TH[crop])
    del arr, ET
    print(f"{s}: {len(SEAS[(s, CROPS[0])][0])} seasons", flush=True)

## 2. Events relative to the reference (75th percentile of the heat indicators, SPEI)

SPEI: the log-logistic distribution of the water balance fitted on the reference seasons of each cell by **L-moments**,
in the generalized-logistic parametrisation of the SPEI package (Beguería et al. 2014, *Int. J. Climatol.* 34,
3001–3023; Hosking 1997, `lmom::pelglo`): k = −τ3, α = λ2 sin(kπ) / (kπ), ξ = λ1 − α (1/k − π / sin(kπ)),
F(D) = 1 / (1 + e^−y), y = −log(1 − k (D − ξ) / α) / k; SPEI = Φ^−1(F). Unlike the probability-weighted moments with
the lower bound of Vicente-Serrano et al. (2010), it also fits water balances with a long dry tail (negative skew),
common in dry climates (checked on synthetic samples: no failed fit; out of sample, with 34 reference seasons,
P(SPEI ≤ −1.5) = 7.4–8.6 % instead of 6.7 % and P(SPEI ≤ −2) = 2.3–2.9 % instead of 2.3 %). The same parameters are
applied to the runs. **Reading**: the reference is in-sample (its frequencies are the nominal ones), the runs are out of
sample and carry about +1 point from the sampling of the fit even without change: the hosing − no hosing differences
are free of it, the runs vs historical differences include it.

In [ ]:
def glo_fit(D):
    # generalized logistic fitted by sample L-moments (Hosking 1997; lmom::pelglo, as the SPEI package): D (seasons,
    # cells) of the reference -> xi, alpha, k per cell (NaN where |tau3| >= 1 or no spread)
    x = np.sort(D, axis=0)
    n = x.shape[0]
    i = np.arange(1, n + 1)[:, None]
    b0 = x.mean(0)
    b1 = (((i - 1) / (n - 1)) * x).mean(0)
    b2 = (((i - 1) * (i - 2) / ((n - 1) * (n - 2))) * x).mean(0)
    l1, l2, l3 = b0, 2 * b1 - b0, 6 * b2 - 6 * b1 + b0
    with np.errstate(invalid="ignore", divide="ignore"):
        k = -l3 / l2
        ok = (l2 > 0) & (np.abs(k) < 1)
        k = np.where(np.abs(k) < 1e-6, 1e-6, k)
        a = l2 * np.sin(k * np.pi) / (k * np.pi)
        xi = l1 - a * (1 / k - np.pi / np.sin(k * np.pi))
    return np.where(ok, xi, np.nan), np.where(ok, a, np.nan), np.where(ok, k, np.nan)


def spei(D, par):
    xi, a, k = par
    with np.errstate(invalid="ignore", divide="ignore", over="ignore"):
        arg = 1 - k * (D - xi) / a
        F = np.where(arg > 0, 1 / (1 + np.exp(np.log(np.where(arg > 0, arg, 1.0)) / k)), np.where(k > 0, 1.0, 0.0))
    return np.where(np.isfinite(xi), norm.ppf(np.clip(F, 1e-6, 1 - 1e-6)), np.nan)


EV, INFO = {}, []                            # (scenario, crop) -> {event: (seasons, cells) bool}
for crop in CROPS:
    _, ref = SEAS[("historical", crop)]
    p_ref = {h[0]: np.percentile(ref[h[0]], HEAT_PCT, axis=0) for h in HEAT[crop]}
    par = {k: glo_fit(ref[k]) for k in ("D_SEASON", "D_P2")}
    w = CROP[crop]["w"]
    INFO.append({"crop": crop, "cells": w.size, "reference seasons": ref["D_P2"].shape[0],
                 **{f"area where the reference P{HEAT_PCT} of '{n}' is 0 (%)": 100 * w[p == 0].sum() / w.sum() for n, p in p_ref.items()},
                 "area without SPEI fit (%)": 100 * w[~np.isfinite(par['D_P2'][0])].sum() / w.sum()})
    for s in SCEN:
        _, x = SEAS[(s, crop)]
        e = {n: x[n] > p[None] for n, p in p_ref.items()}
        for k, lab in (("D_SEASON", "season"), ("D_P2", "phase 2")):
            z = spei(x[k], par[k])
            for thr in SPEI_THR:
                e[f"drought {lab} (SPEI <= {thr})"] = z <= thr
        EV[(s, crop)] = e
INFO = pd.DataFrame(INFO).set_index("crop")
with pd.option_context("display.max_columns", 30, "display.width", 250):
    display(INFO.round(1))

## 3. Frequency of the events on today's cropland (area-weighted), globally and per AR6 region

In [ ]:
def zones_of(crop):
    # (name, cells, weights): today's cropland globally and in its N_REGIONS largest AR6 regions (harvested area), or
    # for 11.d the domains (today's cropland: harvested area; the others: land area)
    c = CROP[crop]
    if RISK_DOMAIN == "suitable":
        return [(k, m, c["w"] if k == "cropland today" else c["wl"]) for k, m in c["dom"].items() if m.any()]
    i, j = c["ij"]
    reg = REG[i, j]
    tot = pd.Series(c["w"]).groupby(reg).sum().sort_values(ascending=False) if REG_NAME else pd.Series(dtype=float)
    top = [int(r) for r in tot.index[:N_REGIONS] if np.isfinite(r)]
    return [("global", np.ones(i.size, bool), c["w"])] + [(REG_NAME[r], reg == r, c["w"]) for r in top]


rows, AREA = [], []
for crop in CROPS:
    zones = zones_of(crop)
    AREA += [{"crop": crop, "zone": zn, "area (Mha)": w[z].sum() / 1e6, "cells": int(z.sum())} for zn, z, w in zones]
    for s in SCEN:
        for ev, x in EV[(s, crop)].items():
            f = x.mean(axis=0)                                     # frequency of the event per cell
            for zname, z, w in zones:
                rows.append({"crop": crop, "zone": zname, "event": ev, "scenario": s,
                             "frequency (%)": 100 * np.average(f[z], weights=w[z])})
AREA = pd.DataFrame(AREA).set_index(["crop", "zone"])
AREA.to_csv(OUT / "risk_zones_area.csv")
FREQ = pd.DataFrame(rows).set_index(["crop", "zone", "event", "scenario"])["frequency (%)"].unstack("scenario")[SCEN]
for h, o in PAIRS:
    FREQ[f"{h} - {o}"] = FREQ[h] - FREQ[o]
FREQ.to_csv(OUT / "risk_frequency.csv")
with pd.option_context("display.max_rows", 500, "display.width", 250):
    display(AREA.round(1))
    if RISK_DOMAIN == "cropland":
        display(FREQ.xs("global", level="zone").round(1))
    display(FREQ.round(1))

## 4. Probability of exceedance of the share of today's cropland affected in a season (Ceglar et al., figure 2)

In [ ]:
COLORS = {"historical": "#2ca02c", "gcm_2025_nohosing": "#ff7f0e", "gcm_2025_hosing": "#1f77b4",
          "gcm_2050_nohosing": "#d62728", "gcm_2050_hosing": "#9467bd"}
SHOW = {c: [h[0] for h in HEAT[c]] + [f"drought phase 2 (SPEI <= {SPEI_THR[0]})"] for c in CROPS}


def exceedance(crop, ev, zone_mask, weights):
    w = weights[zone_mask]
    out = {}
    for s in SCEN:
        share = (EV[(s, crop)][ev][:, zone_mask] * w[None]).sum(1) / w.sum()   # share of the cropland, per season
        out[s] = np.sort(share)[::-1]
    return out


for crop in CROPS:
    zones = zones_of(crop)
    if RISK_DOMAIN == "suitable":                # 11.d: the base domains and those of the last pair
        zones = [z for z in zones if z[0] in ("cropland today", "suitable on ERA5") or any(x in z[0] for x in PAIRS[-1])]
    show = SHOW[crop]
    fig, axes = plt.subplots(len(show), len(zones), figsize=(3.2 * len(zones), 3.0 * len(show)), squeeze=False, sharey=True)
    for r, ev in enumerate(show):
        for k, (zname, z, wz) in enumerate(zones):
            ax = axes[r, k]
            for s, sh in exceedance(crop, ev, z, wz).items():
                ax.plot(sh, np.arange(1, sh.size + 1) / sh.size, color=COLORS.get(s), lw=1.5, label=s)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
            ax.set_title(f"{zname}", fontsize=8 if RISK_DOMAIN == "suitable" else 9)
            if k == 0:
                ax.set_ylabel(f"{ev}\nprobability of exceedance", fontsize=7)
            if r == len(show) - 1:
                ax.set_xlabel("share of the zone (area)", fontsize=8)
    axes[0, -1].legend(fontsize=7, loc="upper right")
    fig.suptitle(f"{crop}: seasons affected (events relative to EC-Earth3 historical {REF_PERIOD[0]}-{REF_PERIOD[1]})", fontsize=10)
    fig.tight_layout(); fig.savefig(OUT / f"exceedance_{crop}.png", dpi=90)
    plt.show() if crop == CROPS[0] else plt.close(fig)

## 5. Maps of the event frequency (maize): 2050 no hosing, 2050 hosing and the difference; heat indicators ERA5 vs EC-Earth3

In [ ]:
crop = "Maize_rf" if "Maize_rf" in CROPS else CROPS[0]
i, j = CROP[crop]["ij"]
pair = PAIRS[-1]


def grid_of(x):
    g = np.full(LAND.shape, np.nan)
    g[i, j] = x
    return g


show = SHOW[crop]
fig, axes = plt.subplots(len(show), 3, figsize=(30, 5.2 * len(show)), squeeze=False)
for r, ev in enumerate(show):
    fo, fh = (100 * EV[(s, crop)][ev].mean(axis=0) for s in (pair[1], pair[0]))
    for k, (val, title, cmap, lim) in enumerate([(fo, pair[1], "magma_r", (0, 100)), (fh, pair[0], "magma_r", (0, 100)),
                                                 (fh - fo, f"{pair[0]} - {pair[1]}", "RdBu_r", (-30, 30))]):
        ax = axes[r, k]
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=4, shading="nearest")
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], grid_of(val), cmap=cmap, vmin=lim[0], vmax=lim[1], shading="nearest")
        ax.set_ylim(-60, 75); ax.set_title(f"{crop}, {ev}: {title} (% of the seasons)", fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(OUT / f"risk_maps_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()

if ERA5_CHECK:                               # ERA5 at the threshold of the literature vs EC-Earth3 at the one used
    rows = []
    for c in CROPS:
        w = CROP[c]["w"]
        for h in HEAT[c]:
            e5, ec = E5[c][h[0]], SEAS[("historical", c)][1][h[0]]
            rows.append({"crop": c, "indicator": h[0], "ERA5, mean per season": np.average(e5.mean(0), weights=w),
                         f"EC-Earth3 historical ({'mapped' if MAPPED else 'absolute'}), mean per season": np.average(ec.mean(0), weights=w),
                         "area never reached in ERA5 (%)": 100 * w[e5.sum(0) == 0].sum() / w.sum(),
                         "area never reached in EC-Earth3 (%)": 100 * w[ec.sum(0) == 0].sum() / w.sum()})
    CHK = pd.DataFrame(rows).set_index(["crop", "indicator"])
    CHK.to_csv(OUT / "hot_days_era5_vs_ecearth3.csv")
    with pd.option_context("display.width", 250):
        display(CHK.round(2))

## 6. Internal variability: is the effect of the collapse larger than the year-to-year noise?

Each run gives one series of seasons: for a zone, the share of its area (weighted as above) affected in each season,
whose mean is the frequency of section 3. The effect of the collapse is the difference of the means of the two runs
(hosing − no hosing, two independent runs). **t-test** with the effective sample size of each series, n_eff = n (1 − r1)
/ (1 + r1), r1 = lag-1 autocorrelation (clipped to 0–0.9), as for autocorrelated climate series (Zwiers & von Storch
1995, *J. Climate* 8, 336–351; Wilks 2011, *Statistical Methods in the Atmospheric Sciences*): p < 0.05 = the
effect is beyond the year-to-year variability of the runs. **Halves**: the effect computed on the first and on the
second half of the seasons of each run; the same sign in both is a second check (a drift inside a run also shows up
there). `risk_significance.csv`: every crop, zone, event and pair.

In [ ]:
from scipy.stats import t as t_dist


def n_eff(x):
    r1 = np.corrcoef(x[:-1], x[1:])[0, 1] if x.size > 3 and x.std() > 0 else 0.0
    r1 = float(np.clip(np.nan_to_num(r1), 0.0, 0.9))
    return x.size * (1 - r1) / (1 + r1)


rows = []
for crop in CROPS:
    for zname, z, wz in zones_of(crop):
        w = wz[z]
        if w.sum() <= 0:
            continue
        for ev in SHOW[crop]:
            sh = {s: 100 * (EV[(s, crop)][ev][:, z] * w[None]).sum(1) / w.sum() for s in SCEN}   # % of the zone, per season
            for h, o in PAIRS:
                a, b = sh[h], sh[o]
                d = a.mean() - b.mean()
                na, nb = n_eff(a), n_eff(b)
                se = np.sqrt(a.var(ddof=1) / na + b.var(ddof=1) / nb)
                pv = 2 * t_dist.sf(abs(d) / se, max(na + nb - 2, 1)) if se > 0 else float(d == 0)
                ka, kb = a.size // 2, b.size // 2
                d1, d2 = a[:ka].mean() - b[:kb].mean(), a[ka:].mean() - b[kb:].mean()
                rows.append({"crop": crop, "zone": zname, "event": ev, "pair": f"{h} - {o}", "effect (points)": d,
                             "SE": se, "p": pv, "first half": d1, "second half": d2,
                             "same sign in both halves": np.sign(d1) == np.sign(d2),
                             "robust": (pv < 0.05) and np.sign(d1) == np.sign(d2),
                             "n_eff hosing": na, "n_eff no hosing": nb})
SIG = pd.DataFrame(rows).set_index(["crop", "zone", "event", "pair"])
SIG.to_csv(OUT / "risk_significance.csv")
last = f"{PAIRS[-1][0]} - {PAIRS[-1][1]}"
summ = []
for lo in (2, 5, 10):
    sel = SIG[SIG["effect (points)"].abs() >= lo]
    summ.append({"|effect| >= (points)": lo, "effects": len(sel), "p < 0.05 (%)": 100 * (sel.p < 0.05).mean(),
                 "same sign in both halves (%)": 100 * sel["same sign in both halves"].mean(),
                 "robust (%)": 100 * sel.robust.mean()})
with pd.option_context("display.max_rows", 300, "display.width", 250, "display.max_colwidth", 50):
    print("Share of the hosing effects that are beyond the year-to-year variability (all crops, zones, events, pairs)")
    display(pd.DataFrame(summ).set_index("|effect| >= (points)").round(1))
    big = SIG.xs(last, level="pair")
    big = big[big["effect (points)"].abs() >= 4].sort_values("effect (points)")
    print(f"{last}: effects of 4 points or more")
    display(big[["effect (points)", "SE", "p", "first half", "second half", "robust"]].round(3))

## 7. Cell by cell: where is the effect of the AMOC weakening beyond the year-to-year variability?

For each crop, event, pair and cell of today's cropland, each season is bad or not: the frequency of the bad seasons
in the hosing and in the original run is compared with **Fisher's exact test** (two-sided; 2 × 2 table of bad /
other seasons of the two runs). Thousands of cells are tested, so about 5 % would be "significant" at p < 0.05 by
chance alone: the **false discovery rate** is controlled with Benjamini & Hochberg (1995) at `FDR_Q` = 0.10, the
level Wilks (2016, *Bull. Amer. Meteor. Soc.* 97, 2263–2273) recommends for maps of spatially correlated fields
(α_FDR = 2 α_global). A cell is a **robust** increase / decrease when its test passes the FDR control and the effect has
the same sign in the first and in the second half of the seasons of the runs (as 12.b for the suitability). Outputs:
`risk_cells_<crop>.nc` (difference of frequency, p, robust flag per event and pair), `risk_cells_areas.csv` (share
of today's cropland with a robust increase / decrease, globally and in the main AR6 regions; share with p < 0.05
before the FDR control), maps of 2050 for every crop (difference where robust, other cells grey).

In [ ]:
from scipy.stats import fisher_exact
FDR_Q = 0.10


def bh_reject(p, q):
    # Benjamini-Hochberg: True where the hypothesis is rejected at the false discovery rate q
    p = np.asarray(p, float)
    ok = np.isfinite(p)
    out = np.zeros(p.shape, bool)
    m = int(ok.sum())
    if m == 0:
        return out
    ps = np.sort(p[ok])
    k = np.nonzero(ps <= q * np.arange(1, m + 1) / m)[0]
    if k.size:
        out[ok] = p[ok] <= ps[k.max()]
    return out


rows, CELLS = [], {}
for crop in CROPS:
    c = CROP[crop]
    i, j = c["ij"]
    dv = {}
    for ev in SHOW[crop]:
        for h, o in PAIRS:
            a, b = EV[(h, crop)][ev], EV[(o, crop)][ev]                  # (seasons, cells) bool
            na, nb = a.shape[0], b.shape[0]
            ka, kb = a.sum(0), b.sum(0)
            p = np.ones(ka.size)
            for k in np.nonzero((ka + kb > 0) & (ka + kb < na + nb))[0]:
                p[k] = fisher_exact([[ka[k], na - ka[k]], [kb[k], nb - kb[k]]])[1]
            d = 100 * (ka / na - kb / nb)                                  # points of % of the seasons
            h1 = a[:na // 2].mean(0) - b[:nb // 2].mean(0)
            h2 = a[na // 2:].mean(0) - b[nb // 2:].mean(0)
            same = (np.sign(h1) == np.sign(h2)) & (np.sign(h1) == np.sign(d)) & (d != 0)
            robust = bh_reject(p, FDR_Q) & same
            tag = f"{ev.split(' (')[0].replace(' ', '_')}_{h.split('_')[1] if '_' in h else h}"
            for name, x in (("diff", d), ("p", p), ("robust", robust.astype(float))):
                g = np.full(LAND.shape, np.nan, np.float32); g[i, j] = x
                dv[f"{name}_{tag}"] = (("lat", "lon"), g)
            for zname, z, wz in zones_of(crop):
                tot = wz[z].sum()
                rows.append({"crop": crop, "event": ev, "pair": f"{h} - {o}", "zone": zname, "cells": int(z.sum()),
                             "robust increase (% of the area)": 100 * wz[z & robust & (d > 0)].sum() / tot,
                             "robust decrease (% of the area)": 100 * wz[z & robust & (d < 0)].sum() / tot,
                             "p < 0.05 before FDR (% of the area)": 100 * wz[z & (p < 0.05)].sum() / tot,
                             "mean change (points)": float(np.average(d[z], weights=wz[z]))})
            CELLS[(crop, ev, h)] = (d, robust)
    xr.Dataset(dv, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "test": "Fisher exact per cell, Benjamini-Hochberg FDR q = 0.10, same sign in both halves"}
               ).to_netcdf(OUT / f"risk_cells_{crop}.nc")
CA = pd.DataFrame(rows).set_index(["crop", "event", "pair", "zone"])
CA.to_csv(OUT / "risk_cells_areas.csv")
last = f"{PAIRS[-1][0]} - {PAIRS[-1][1]}"
with pd.option_context("display.max_rows", 400, "display.width", 250, "display.max_colwidth", 45):
    print(f"{last}: share of today's cropland with a robust change of the bad seasons (global)")
    display(CA.xs((last, "global"), level=("pair", "zone")).round(1))
    print(f"{last}: by region")
    display(CA.xs(last, level="pair").round(1))

h2050 = PAIRS[-1][0]
for crop in CROPS:
    i, j = CROP[crop]["ij"]
    evs = SHOW[crop]
    fig, axes = plt.subplots(1, len(evs), figsize=(10 * len(evs), 5.2), squeeze=False)
    for ax, ev in zip(axes[0], evs):
        d, rb = CELLS[(crop, ev, h2050)]
        g_all = np.full(LAND.shape, np.nan); g_all[i, j] = 0.0
        g = np.full(LAND.shape, np.nan); g[i, j] = np.where(rb, d, np.nan)
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=4, shading="nearest")
        ax.pcolormesh(GRID["lon"], GRID["lat"], g_all, cmap="Greys", vmin=-1, vmax=1.5, shading="nearest")
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], g, cmap="RdBu_r", vmin=-30, vmax=30, shading="nearest")
        a = CA.loc[(crop, ev, f"{h2050} - {PAIRS[-1][1]}", "global")]
        ax.set_ylim(-60, 75)
        ax.set_title(f"{crop}, {ev}, 2050 hosing - no hosing, robust cells only (points)\nrobust increase "
                     f"{a['robust increase (% of the area)']:.1f} %, decrease {a['robust decrease (% of the area)']:.1f} % of today's cropland", fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout(); fig.savefig(OUT / f"risk_cells_maps_{crop}.png", dpi=90, bbox_inches="tight")
    plt.show() if crop == CROPS[0] else plt.close(fig)